# Part 1 : Practise

## Task 1 : Load it, remove the empty columns

### Load the file and make a copy

In [1]:
import pandas as pd

rentals_raw = pd.read_csv("data/bike_rentals.csv")
rentals = rentals_raw.copy()

### print the shape

In [2]:
rentals.shape

(403, 15)

### Find the data type of all columns

In [3]:
rentals.dtypes

rental_id            str
station              str
bike_type            str
helmets_text         str
is_member            str
insured              str
paid_online          str
status               str
coupon_code      float64
damage_notes     float64
late_fee_note    float64
start_date           str
end_date             str
price                str
rating           float64
dtype: object

### Find the empty columns

In [4]:
rentals.columns[rentals.isna().all()]

Index(['coupon_code', 'damage_notes', 'late_fee_note'], dtype='str')

- isna() puts True wherever a value is missing
- all() asks for each column is every vale True?
- rentals.columns[...] keeps only the column names where the answer was True

### Remove the empty columns

In [7]:
rentals = rentals.dropna(axis=1, how="all")

### dropna()

By default, `dropna()` removes the **rows** that have at least one missing value.

These two lines are the same:

    rentals.dropna()
    rentals.dropna(axis=0, how='any')

- `axis=0` means rows
- `how='any'` means remove the row if even one value is missing

This one removes the **columns** where the whole column is empty:

    rentals.dropna(axis=1, how='all')

- `axis=1` means columns
- `how='all'` means remove the column only if all its values are missing

In [8]:
rentals.shape

(403, 12)

In [9]:
rentals.columns[rentals.isna().all()]

Index([], dtype='str')

**Answer:** 
- The table has 403 rows and 15 columns. 
- The price column is text (str), not a number.
- I found 3 empty columns: coupon_code, damage_notes and late_fee_note. 
- After I removed them, 12 columns are left and 0 columns are empty.

## Task 2 : The price: from text to numbers

### Step 1 : Remove Sek

In [12]:
rentals["price"] = rentals["price"].str.replace("SEK ", "", regex=False)

In [13]:
rentals["price"].head(3)

0      855.50
1    1,311.50
2      350.00
Name: price, dtype: str

### Step 2 : Remove the comma

In [14]:
rentals["price"] = rentals["price"].str.replace(",", "", regex=False)

In [15]:
rentals["price"].head(3)

0     855.50
1    1311.50
2     350.00
Name: price, dtype: str

### Change the type from str to float

In [16]:
rentals["price"] = rentals["price"].astype(float)

In [20]:
rentals["price"].dtype

dtype('float64')

In [19]:
rentals["price"].describe()

count     352.000000
mean      847.348011
std       633.435513
min       154.500000
25%       406.000000
50%       668.250000
75%      1194.625000
max      3401.000000
Name: price, dtype: float64

### Three most expensive rentals for columns (rental_id, bike_type, status and price)


In [21]:
rentals.sort_values("price", ascending=False).head(3)[["rental_id", "bike_type", "status", "price"]]

,rental_id,bike_type,status,price
187,R-5282,Cargo bike,Completed,3401.0
49,R-5209,Cargo bike,Completed,2979.5
270,R-5378,E-bike,Completed,2941.0


**Answer:** 
- The price was text like "SEK 1,311.50". 
- First I removed "SEK " and then the comma.
- After that I changed it to float. 
- 352 rentals have a price. 
- The mean is 847.3, the median is 668.2, the lowest price is 154.5 and the highest is 3401.0. 
- The three most expensive rentals are R-5282, R-5209 and R-5378.

## Task 3 : The missing Prices

### Find how many prices are missing and it's percent

In [22]:
rentals["price"].isna().sum()

np.int64(51)

In [24]:
(rentals["price"].isna().mean()*100).round(1)

np.float64(12.7)

In [25]:
pd.crosstab(rentals["price"].isna(), rentals["status"] == "Cancelled")

status,False,True
price,,
False,352,0
True,4,47


**Answer:** 
- 51 prices are missing, which is 12.7 %.
- Among the 51 missing prices, 47 of them are cancelled rentals, so they are missing for a reason: nothing was paid. 
- 4 are not cancelled, so they are missing by accident.
- I will leave all 51 as missing, because I don't know the real prices. 

## Task 4 : The letters t and f

### Find the columns that contain only t and f

In [35]:
tf_cols = [c for c in rentals.columns if set(rentals[c].dropna().unique()) <= {"t", "f"}]
tf_cols

['is_member', 'insured', 'paid_online']

In [33]:
rentals["status"].unique()

<StringArray>
['Completed', 'No-show', 'Cancelled']
Length: 3, dtype: str

- "status" is not in the list because it doesn't have t and f values.
- It has 'Completed', 'No-show', 'Cancelled' values 

### Change the columns to True and False with map

In [36]:
for c in tf_cols:
    rentals[c] = rentals[c].map({"t": True, "f": False})

### How many missing values does each column have afterwords

In [37]:
rentals[tf_cols].isna().sum()

is_member      0
insured        3
paid_online    0
dtype: int64

### Compare median Price of members and non-members

In [38]:
rentals["is_member"].value_counts()

is_member
False    288
True     115
Name: count, dtype: int64

In [40]:
rentals.groupby("is_member")["price"].agg(["median", "count"])

,median,count
is_member,,
False,756.25,248
True,558.00,104


**Answer:** 
- The columns with only t and f are is_member, insured and paid_online. 
- status is not in the list because it has the words Completed, Cancelled and No-show. 
- After map, insured has 3 missing values and the other two have 0. 
- There are 288 non-members and 115 members. 
- The median price is 756 for non-members (248 rentals) and 558 for members (104 rentals). 
- Members pay less, but I can't say yet that being a member is the reason.